# canteen_checkout on Colab

准备工作（只需一次）：
1. 把 `canteen_checkout.zip` 和 `unimib_yolo.zip` 上传到 Google Drive 的 `MyDrive/canteen/`
2. 菜单 **代码执行程序 → 更改运行时类型 → T4 GPU**

数据解压到 Colab 本地磁盘 `/content/work`（读得快）；所有输出（权重、gallery、报告）直接写到 `MyDrive/canteen/runs/`，断线也不会丢。

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob, json
DRIVE = '/content/drive/MyDrive/canteen'
RUNS  = f'{DRIVE}/runs'
WORK  = '/content/work'
os.makedirs(RUNS, exist_ok=True)
os.makedirs(WORK, exist_ok=True)

# 保持和 Mac 上一样的目录关系：canteen_checkout/ 与 unimib_yolo/ 并列
for name in ('canteen_checkout', 'unimib_yolo'):
    if not os.path.isdir(f'{WORK}/{name}'):
        !cp "{DRIVE}/{name}.zip" /content/ && unzip -q "/content/{name}.zip" -d "{WORK}" && rm "/content/{name}.zip"
DATA = f'{WORK}/unimib_yolo'
%cd {WORK}/canteen_checkout
!ls {DATA}

In [ ]:
# Colab 自带 torch/torchvision/numpy/opencv/scipy，不要重装它们
!pip install -q ultralytics timm faiss-cpu ensemble-boxes
!python tests/smoke_test.py 2>&1 | tail -4

## 1. 训练检测器

Colab 只有 2 个 CPU，`--workers 2`。T4 上 `yolo11s-seg` + batch 16 没问题；想先快速出 baseline 就把 `MODEL` 改成 `yolo11n-seg.pt`。

In [ ]:
MODEL = 'yolo11s-seg.pt'
!python train_detector.py --data {DATA}/data.yaml --model {MODEL} \
    --device 0 --batch 16 --workers 2 --epochs 100 --imgsz 640 \
    --project {RUNS}/detector --name unimib

**断线续训**：如果运行时断开，重新执行上面的 setup 单元格（前 3 个），然后运行下面这个。

In [ ]:
last = max(glob.glob(f'{RUNS}/detector/unimib*/weights/last.pt'), key=os.path.getmtime)
print('resume from', last)
!python train_detector.py --data {DATA}/data.yaml --model "{last}" --resume --device 0 --workers 2

## 2. 构建 gallery（预训练 DINOv2 ViT-S/14）

In [ ]:
!python build_gallery.py --crops {DATA}/crops/train --out {RUNS}/gallery.npz --device cuda

## 3. 在 val 上标定阈值（GT 框），再在 test 上端到端评估

只在 val 上调阈值，test 只用来报告。

In [ ]:
!python evaluate.py --dataset {DATA} --gallery {RUNS}/gallery.npz --detector oracle \
    --split val --calibrate 0.98 --device cuda --out {RUNS}/report_val_calib.json > /dev/null
cal = json.load(open(f'{RUNS}/report_val_calib.json'))['calibration']['suggested']
print(cal)
ACCEPT_SIM, MARGIN = cal['accept_sim'], cal['margin']

In [ ]:
BEST = max(glob.glob(f'{RUNS}/detector/unimib*/weights/best.pt'), key=os.path.getmtime)
print('detector:', BEST)
!python evaluate.py --dataset {DATA} --gallery {RUNS}/gallery.npz --detector "{BEST}" \
    --split test --accept-sim {ACCEPT_SIM} --margin {MARGIN} --device cuda \
    --out {RUNS}/report_test.json --save-vis {RUNS}/vis_test > /dev/null

def summary(path):
    r = json.load(open(path))
    print(json.dumps({'detection': r['detection'],
                      'top1': r['recognition_on_gt_crops']['top1'],
                      'precision_of_accepted': r['recognition_on_gt_crops']['precision_of_accepted'],
                      'end_to_end': r['end_to_end'],
                      'thresholds': r['thresholds']}, indent=2, ensure_ascii=False))
summary(f'{RUNS}/report_test.json')

## 4.（可选）ArcFace 微调 embedder，重建 gallery，重新标定和评估

In [ ]:
!python finetune_embedder.py --train {DATA}/crops/train --val {DATA}/crops/val \
    --out {RUNS}/embedder --balanced --device cuda --workers 2

In [ ]:
CKPT = f'{RUNS}/embedder/best.pt'
!python build_gallery.py --crops {DATA}/crops/train --out {RUNS}/gallery_ft.npz --checkpoint "{CKPT}" --device cuda
!python evaluate.py --dataset {DATA} --gallery {RUNS}/gallery_ft.npz --detector oracle \
    --split val --calibrate 0.98 --device cuda --out {RUNS}/report_val_calib_ft.json > /dev/null
cal = json.load(open(f'{RUNS}/report_val_calib_ft.json'))['calibration']['suggested']
print(cal)
!python evaluate.py --dataset {DATA} --gallery {RUNS}/gallery_ft.npz --detector "{BEST}" \
    --split test --accept-sim {cal['accept_sim']} --margin {cal['margin']} --device cuda \
    --out {RUNS}/report_test_ft.json --save-vis {RUNS}/vis_test_ft > /dev/null
summary(f'{RUNS}/report_test_ft.json')

## 5. 用训练好的模型测试新图片

需要先完成第 1–3 步（或者断线后重跑前 3 个 setup 单元格即可，权重和 gallery 都在 Drive 上）。
想用第 4 步微调过的 embedder，就把 `USE_FT` 改成 `True`。价格表可选：把填好价格的 `prices.csv` 放到 `MyDrive/canteen/`（格式见 `unimib_yolo/prices_template.csv`），没有也能跑。

In [ ]:
import yaml
from google.colab import files
from IPython.display import Image, display

USE_FT = False
BEST = max(glob.glob(f'{RUNS}/detector/unimib*/weights/best.pt'), key=os.path.getmtime)
tag = '_ft' if USE_FT else ''
cal = json.load(open(f'{RUNS}/report_val_calib{tag}.json'))['calibration']['suggested']

cfg = yaml.safe_load(open('configs/checkout.yaml'))
cfg['detector']['weights'] = BEST
cfg['index']['path'] = f'{RUNS}/gallery{tag}.npz'
cfg['embedder']['checkpoint'] = f'{RUNS}/embedder/best.pt' if USE_FT else None
cfg['decision'].update(accept_sim=cal['accept_sim'], margin=cal['margin'])
cfg['pricing']['csv'] = f'{DRIVE}/prices.csv'
CFG = f'{RUNS}/checkout_colab{tag}.yaml'
yaml.safe_dump(cfg, open(CFG, 'w'), allow_unicode=True, sort_keys=False)
print(open(CFG).read())

In [ ]:
# 把照片放进 Drive 的 MyDrive/canteen/new_images/（网页版 Drive 直接拖进去即可）
# 不用 files.upload()：图片稍大或用 Safari 时会报 "Maximum call stack size exceeded"
NEW = f'{DRIVE}/new_images'
os.makedirs(NEW, exist_ok=True)
imgs = sorted(p for p in glob.glob(f'{NEW}/*') if p.lower().endswith(('.jpg', '.jpeg', '.png')))
print(len(imgs), 'images:', [os.path.basename(p) for p in imgs])
if not imgs:
    print('new_images/ 里的文件:', os.listdir(NEW))
    print('Drive 里其他位置的图片:', glob.glob(f'{DRIVE}/**/*.*', recursive=True)[:20])
    raise SystemExit('没找到 .jpg/.png：检查文件夹位置和扩展名；刚拖进 Drive 的文件可能要等一会儿，或重新 mount')

# WEIGHT = 540   # 可选：称重（克），加上 --weight {WEIGHT} 启用重量交叉校验
!python demo.py --config "{CFG}" --image "{NEW}" --save {RUNS}/demo
for p in imgs:
    display(Image(f'{RUNS}/demo/{os.path.basename(p)}', width=800))

## 6. 添加新菜 / 给已有类别补充照片（不需要重新训练）

识别是 kNN 检索，加类别 = 把新照片的 embedding 追加进 gallery。照片放在 `MyDrive/canteen/new_dishes/<类别名>/`，
文件夹名就是类别名。已有类别（如 `banane`）用同名文件夹就是补充照片；新名字会新建类别。
结果写到新文件 `gallery_plus.npz`，原 gallery 保留做对照。

In [ ]:
NEW_DISHES = f'{DRIVE}/new_dishes'
BEST = max(glob.glob(f'{RUNS}/detector/unimib*/weights/best.pt'), key=os.path.getmtime)
for d in sorted(os.listdir(NEW_DISHES)):
    print(d, len(os.listdir(f'{NEW_DISHES}/{d}')), 'files')

# --detector: 用训练好的检测器裁剪（保留最大的检测框），和运行时的裁剪方式一致
# 如果 gallery 是微调版，改成 --append {RUNS}/gallery_ft.npz 并加 --checkpoint {RUNS}/embedder/best.pt
!python build_gallery.py --append {RUNS}/gallery.npz --crops {NEW_DISHES} \
    --out {RUNS}/gallery_plus.npz --detector "{BEST}" --device cuda

In [ ]:
# 用新 gallery 重新生成第 5 节的配置，然后重跑第 5 节的最后一个单元格
cfg = yaml.safe_load(open(CFG))
cfg['index']['path'] = f'{RUNS}/gallery_plus.npz'
CFG = f'{RUNS}/checkout_colab_plus.yaml'
yaml.safe_dump(cfg, open(CFG, 'w'), allow_unicode=True, sort_keys=False)
print('now using', CFG)